# Kaza — Phase 1 Training (Twi + Optional Igbo)

Pre-tokenizes audio to compact `uint16` caches on Google Drive (~1 MB/hour), trains Kaza (485M trainable params: embeddings + 6 new Global GQA layers + MaskGIT depth decoder + MTP), logs held-out `k = 12` validation metrics, auto-resumes checkpoints, and plays periodic Twi continuation samples.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!test -d /content/kaza && git -C /content/kaza pull --ff-only origin phase1-design || git clone -b phase1-design https://github.com/udohchuks/kaza.git /content/kaza
%cd /content/kaza
!pip install -q -r requirements.txt

In [ ]:
# Training & dataset configuration
from pathlib import Path
from kaza.train import TrainConfig

DRIVE_ROOT = Path("/content/drive/MyDrive/kaza_phase1")
RUN_NAME = "twi_v1"

# Dataset acquisition
MAX_TRAIN_HOURS = 10.0   # increase when scaling up (10h ~ 11 MB cached on Drive)
MAX_VAL_HOURS = 0.5
INCLUDE_IGBO = False     # set True to mix google/WaxalNLP ibo_tts into train

# Regularizers
DROP_PROB = 0.08         # random slot zero-out on agent t-1
MTP_OFFSETS = (2, 4)     # training-only t+2 (160ms) & t+4 (320ms)
MTP_WEIGHT = 0.2

cfg = TrainConfig(
    out_dir=str(DRIVE_ROOT / "runs" / RUN_NAME),
    seq_len=375,             # 30s windows at 12.5 Hz
    batch_size=2,            # L4: 2 | A100-40GB: 4 | A100-80GB: 8
    grad_accum=8,            # effective batch = batch_size * grad_accum
    max_steps=2000,
    warmup_freeze_steps=200,
    lr_warmup_steps=100,
    lr=3e-4,
    min_lr=3e-5,
    log_every=10,
    eval_every=100,
    save_every=200,
    sample_every=200,
)
print(f"Run dir: {cfg.out_dir}")

In [ ]:
# Dataset acquisition & pre-tokenization (skips automatically if cached on Drive)
from kaza.env import print_env
from kaza.codec import KazaCodec
from kaza.data import IGBO_TRAIN_SOURCES, TWI_TRAIN_SOURCES, TWI_VAL_SOURCES, build_token_cache

env = print_env()
codec = KazaCodec(device=env.device, dtype=env.frozen_dtype)

train_sources = list(TWI_TRAIN_SOURCES) + (list(IGBO_TRAIN_SOURCES) if INCLUDE_IGBO else [])
tag = "twi_igbo" if INCLUDE_IGBO else "twi"

train_cache = build_token_cache(
    train_sources,
    codec=codec,
    out_path=DRIVE_ROOT / "data" / f"{tag}_train_{int(MAX_TRAIN_HOURS)}h.pt",
    max_hours=MAX_TRAIN_HOURS,
)
val_cache = build_token_cache(
    TWI_VAL_SOURCES,
    codec=codec,
    out_path=DRIVE_ROOT / "data" / f"twi_val_{MAX_VAL_HOURS}h.pt",
    max_hours=MAX_VAL_HOURS,
)
train_tokens = train_cache["tokens"]
val_tokens = val_cache["tokens"]

In [ ]:
# Build KazaModel (expanded 30-layer LFM3 + MaskGIT + MTP + slot corruption)
from kaza.model import KazaModel

model = KazaModel(
    device=env.device,
    frozen_dtype=env.frozen_dtype,
    trainable_dtype=env.trainable_dtype,
    autocast_dtype=env.autocast_dtype,
    drop_prob=DROP_PROB,
    mtp_offsets=MTP_OFFSETS,
    mtp_weight=MTP_WEIGHT,
)
n_train = sum(p.numel() for p in model.trainable_parameters()) / 1e6
print(f"Trainable params (incl. MTP): {n_train:.1f}M")

In [ ]:
# Run Phase 1 training (auto-resumes from latest.pt on Drive if interrupted)
from kaza.train import train_phase1

history = train_phase1(
    model,
    train_tokens=train_tokens,
    val_tokens=val_tokens,
    cfg=cfg,
    codec=codec,
    resume=True,
)

In [ ]:
# Plot loss/accuracy curves and listen to best checkpoint vs held-out reference
import torch
from IPython.display import Audio as Play, display
from kaza.codec import SAMPLE_RATE
from kaza.train import plot_metrics, sample_continuation

plot_metrics(cfg.out_dir)

best_path = Path(cfg.out_dir) / "best.pt"
if best_path.exists():
    ckpt = torch.load(best_path, map_location="cpu", weights_only=False)
    model.load_trainable_state_dict(ckpt["model"])
    print(f"Loaded best.pt (step {ckpt['step']}, val_k12_loss={ckpt['best_val_loss']:.4f})")

ref_wav = Path(cfg.out_dir) / "samples" / "val_reference.wav"
if ref_wav.exists():
    print("Held-out Twi reference (8s):")
    display(Play(str(ref_wav), rate=SAMPLE_RATE))

sample_continuation(model, codec, val_tokens, Path(cfg.out_dir), step=999999, temperature=0.8)